# Proyecto Final – Agente RL para Connect-4
**Curso:** Fundamentos de Inteligencia Artificial – Universidad de La Sabana, 2026.1  
**Rama:** Martin-Jerez

In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt
from rl_agent import QLearningAgent

ROWS, COLS = 6, 7

def apply_move(board, col, player):
    new_board = board.copy()
    for r in range(ROWS - 1, -1, -1):
        if new_board[r, col] == 0:
            new_board[r, col] = player
            break
    return new_board

def check_winner(board):
    for r in range(ROWS):
        for c in range(COLS):
            p = board[r, c]
            if p == 0:
                continue
            if c + 3 < COLS and all(board[r, c+i] == p for i in range(4)): return p
            if r + 3 < ROWS and all(board[r+i, c] == p for i in range(4)): return p
            if r + 3 < ROWS and c + 3 < COLS and all(board[r+i, c+i] == p for i in range(4)): return p
            if r + 3 < ROWS and c - 3 >= 0 and all(board[r+i, c-i] == p for i in range(4)): return p
    return 0

def random_act(board):
    free = [c for c in range(COLS) if board[0, c] == 0]
    return random.choice(free) if free else 3

def play_game(agent_a, agent_b, a_is_minus1=True):
    """Devuelve 1 si gana agent_a, -1 si pierde, 0 si empata."""
    board = np.zeros((ROWS, COLS), dtype=int)
    a_player = -1 if a_is_minus1 else 1
    b_player = -a_player
    current = -1
    while True:
        free = [c for c in range(COLS) if board[0, c] == 0]
        if not free:
            return 0
        if current == a_player:
            col = agent_a(board) if callable(agent_a) else agent_a.act(board)
        else:
            col = agent_b(board) if callable(agent_b) else agent_b.act(board)
        board = apply_move(board, col, current)
        winner = check_winner(board)
        if winner != 0:
            return 1 if winner == a_player else -1
        current = -current

def run_series(agent, opponent_fn, n_games=100):
    wins = draws = losses = 0
    for i in range(n_games):
        r = play_game(agent, opponent_fn, a_is_minus1=(i % 2 == 0))
        if r == 1: wins += 1
        elif r == 0: draws += 1
        else: losses += 1
    return wins, draws, losses

print('Utilidades cargadas correctamente.')

## Experimento 1: Impacto de `n_episodes` vs. agente aleatorio

Entrenamos el `QLearningAgent` con distintos presupuestos de episodios y medimos la tasa de victorias contra un agente que elige columnas al azar.

In [ ]:
episode_values = [100, 500, 1000, 2000, 5000]
win_rates = []

for n_ep in episode_values:
    print(f'Entrenando con n_episodes={n_ep}...')
    agent = QLearningAgent(player=-1, n_episodes=n_ep)
    agent.mount()
    wins, draws, losses = run_series(agent, random_act, n_games=100)
    wr = wins / 100
    win_rates.append(wr)
    print(f'  W={wins} D={draws} L={losses}  (win rate={wr:.2%})')

plt.figure(figsize=(8, 4))
plt.plot(episode_values, [w*100 for w in win_rates], marker='o', linewidth=2)
plt.axhline(50, color='red', linestyle='--', label='Umbral 50%')
plt.xlabel('n_episodes (presupuesto de entrenamiento)')
plt.ylabel('Tasa de victorias (%)')
plt.title('QLearningAgent vs. Agente Aleatorio')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Experimento 2: Auto-desempeño – RL fuerte vs. RL débil

Enfrentamos el agente con 5000 episodios contra uno entrenado con solo 500.

In [ ]:
print('Entrenando agente fuerte (5000 ep)...')
strong = QLearningAgent(player=-1, n_episodes=5000)
strong.mount()

print('Entrenando agente débil (500 ep)...')
weak = QLearningAgent(player=1, n_episodes=500)
weak.mount()

wins, draws, losses = run_series(strong, weak.act, n_games=100)
print(f'Strong vs Weak → W={wins} D={draws} L={losses}')

labels = ['Victorias\n(strong)', 'Empates', 'Derrotas\n(strong)']
values = [wins, draws, losses]
colors = ['#4CAF50', '#FFC107', '#F44336']
plt.figure(figsize=(6, 4))
plt.bar(labels, values, color=colors)
plt.title('RL-5000ep vs RL-500ep (100 partidas)')
plt.ylabel('Número de partidas')
plt.grid(axis='y')
plt.tight_layout()
plt.show()

## Experimento 3: QLearningAgent (5000 ep) vs. MCTSAgentRandom (200 sims)

Comparamos el aprendizaje offline contra la búsqueda online del agente base del `master`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
from mcts_random import MCTSAgentRandom

print('Entrenando QLearningAgent (5000 ep)...')
ql_agent = QLearningAgent(player=-1, n_episodes=5000)
ql_agent.mount()

mcts_agent = MCTSAgentRandom(player=1, num_simulations=200)
mcts_agent.mount()

wins, draws, losses = run_series(ql_agent, mcts_agent.act, n_games=50)
print(f'QL vs MCTS-200 → W={wins} D={draws} L={losses}')

labels = ['QL gana', 'Empate', 'MCTS gana']
values = [wins, draws, losses]
colors = ['#2196F3', '#FFC107', '#FF5722']
plt.figure(figsize=(6, 4))
plt.bar(labels, values, color=colors)
plt.title('QLearningAgent (5000 ep) vs MCTSAgentRandom (200 sims)\n50 partidas')
plt.ylabel('Número de partidas')
plt.grid(axis='y')
plt.tight_layout()
plt.show()

## Conclusiones

### Aprendizaje offline vs. búsqueda online (MCTS)

| Aspecto | MCTS (búsqueda online) | QLearningAgent (aprendizaje offline) |
|---------|------------------------|--------------------------------------|
| Tiempo por movimiento | O(simulaciones) | O(1) – lookup de features |
| Memoria | O(árbol de juego) | O(n_features) – vector de 8 pesos |
| Mejora con tiempo extra | Sí (más simulaciones) | No (fijo tras entrenamiento) |
| Generalización | Solo el estado actual | Transfiere vía features |
| Fundamento teórico | Diapo 13 (Online Policy Improvement) | Diapo 12 (Competitive MDPs) + Cross-MDP PI |

### Observaciones

1. **`n_episodes` como variable principal**: más episodios de auto-juego se traducen directamente en mejor rendimiento frente al agente aleatorio, análogo a aumentar el presupuesto de simulaciones en MCTS.

2. **Truco bipolar**: al propagar recompensas hacia atrás negando el valor en cada cambio de turno (`r = -γ·r`), el agente aprende una sola Q-function válida para ambos jugadores, tal como se formaliza en la Diapositiva 12.

3. **Ventaja en tiempo de inferencia**: a diferencia de MCTS, el `QLearningAgent` actúa en tiempo constante durante la partida real; el costo computacional se paga por adelantado en el entrenamiento.

4. **Limitación**: la aproximación lineal de Q con 8 features puede no capturar toda la riqueza táctica de Connect-4 con más de 4.5 × 10¹² estados. Una red neuronal (DQN) sería el paso natural para mejorar la representación.